In [27]:
import pandas as pd
import plotly.express as px

# 1. Load the merged dataset from excel
xls = pd.ExcelFile('merged.xlsx')
df_merged = pd.read_excel(xls, xls.sheet_names[0])

# Clean string columns
df_merged['SECTOR'] = df_merged['SECTOR'].astype(str).str.strip()
df_merged['TYPE'] = df_merged['TYPE'].astype(str).str.strip()

# 2. Data Preparation & Calculations
df_merged['COGS'] = df_merged['HARDWARE'] + df_merged['SOFTWARE'] + df_merged['MANPOWER']
df_merged['GROSS_PROFIT'] = df_merged['REVENUE'] - df_merged['COGS']

# Group data by Industry Sector and Client Type
sector_type_df = df_merged.groupby(['SECTOR', 'TYPE'], as_index=False).agg({
    'GROSS_PROFIT': 'sum',
    'REVENUE': 'sum'
})
sector_type_df['GROSS_MARGIN'] = (sector_type_df['GROSS_PROFIT'] / sector_type_df['REVENUE']) * 100

# 3. Create Plotly Express Bar Chart
fig_px = px.bar(
    sector_type_df,
    x='SECTOR',
    y='GROSS_PROFIT',
    color='TYPE',
    barmode='group',
    text_auto='.2s', # Automatically shows values on bars
    hover_data=['GROSS_MARGIN', 'REVENUE'],
    title='<b>Gross Profit by Industry Sector and Client Type</b>',
    labels={
        'SECTOR': 'Industry Sector', 
        'GROSS_PROFIT': 'Total Gross Profit ($)', 
        'TYPE': 'Client Type'
    }
)

# 4. Add Interactive Scale Toggle (Linear vs. Logarithmic to see NPO values clearly)
fig_px.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            active=0,
            x=0.85,
            y=1.15,
            buttons=list([
                dict(label="Linear Scale",
                     method="relayout",
                     args=[{"yaxis.type": "linear"}]),
                dict(label="Log Scale (Inspect NPOs)",
                     method="relayout",
                     args=[{"yaxis.type": "log"}])
            ]),
        )
    ],
    template='plotly_white', 
    xaxis_tickangle=-45,
    yaxis_title='Total Gross Profit ($)'
)

fig_px.show()

In [26]:
import pandas as pd
import plotly.graph_objects as go

# 1. Load data (assuming your file is named merged.xlsx)
xls = pd.ExcelFile('merged.xlsx')
df = pd.read_excel(xls, xls.sheet_names[0])

# 2. Financial Calculations
df['COGS'] = df['HARDWARE'] + df['SOFTWARE'] + df['MANPOWER']
df['GROSS_PROFIT'] = df['REVENUE'] - df['COGS']

# 3. Map Organization Size from text ranges in 'STAFF STRENGTH'
def parse_org_size(val):
    if pd.isna(val): return 'Unknown'
    val_str = str(val).strip()
    if '~' in val_str:
        parts = val_str.split('~')
        try:
            return 'Small' if float(parts[1].strip()) <= 49 else 'Medium'
        except ValueError: return 'Unknown'
    elif '>' in val_str or '200' in val_str:
        return 'Large'
    return 'Large'

df['ORG_SIZE'] = df['STAFF STRENGTH'].apply(parse_org_size)

# 4. Correctly Map Location using the 'COUNTRY' column
df['LOCATION'] = df['COUNTRY'].apply(
    lambda x: 'Singapore' if str(x).strip().title() == 'Singapore' else 'Overseas'
)

# 5. Group data by Organization Size and Location
size_loc_df = df.groupby(['ORG_SIZE', 'LOCATION'], as_index=False).agg({
    'GROSS_PROFIT': 'sum'
})

sizes = ['Small', 'Medium', 'Large']
fig = go.Figure()

# Add traces for Singapore and Overseas
for loc in ['Singapore', 'Overseas']:
    loc_data = size_loc_df[size_loc_df['LOCATION'] == loc]
    y_vals = [
        loc_data[loc_data['ORG_SIZE'] == s]['GROSS_PROFIT'].values[0] 
        if s in loc_data['ORG_SIZE'].values else 0 
        for s in sizes
    ]
    
    fig.add_trace(go.Bar(
        name=loc,
        x=sizes,
        y=y_vals,
        text=[f"${val:,.0f}" if val > 0 else "" for val in y_vals],
        textposition='auto',
        visible=(loc == 'Singapore') # Default view shows Singapore
    ))

# Add combined 'All Locations' trace
all_data = size_loc_df.groupby('ORG_SIZE', as_index=False)['GROSS_PROFIT'].sum()
y_vals_all = [
    all_data[all_data['ORG_SIZE'] == s]['GROSS_PROFIT'].values[0] 
    if s in all_data['ORG_SIZE'].values else 0 
    for s in sizes
]
fig.add_trace(go.Bar(
    name='All Locations',
    x=sizes,
    y=y_vals_all,
    text=[f"${val:,.0f}" if val > 0 else "" for val in y_vals_all],
    textposition='auto',
    visible=False
))

# 6. Add Interactive Dropdown Menu (Updatemenus)
fig.update_layout(
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            active=0,
            x=0.17,
            y=1.15,
            buttons=list([
                dict(label="Singapore Only",
                     method="update",
                     args=[{"visible": [True, False, False]},
                           {"title": "<b>Gross Profit by Org Size (Singapore)</b>"}]),
                dict(label="Overseas Only",
                     method="update",
                     args=[{"visible": [False, True, False]},
                           {"title": "<b>Gross Profit by Org Size (Overseas)</b>"}]),
                dict(label="All Locations Combined",
                     method="update",
                     args=[{"visible": [False, False, True]},
                           {"title": "<b>Gross Profit by Org Size (All Locations)</b>"}])
            ]),
        )
    ],
    title='<b>Gross Profit Comparison by Organization Size (Singapore)</b>',
    xaxis_title='Organization Size',
    yaxis_title='Total Gross Profit ($)',
    barmode='group',
    template='plotly_white'
)

fig.show()